# Pattern 1: Sequential workflow (prompt chaining)

**Module 3: Agent Design Patterns** of the [Agentic AI Crash Course](https://agenticaikfupm.github.io/CrashCourse/patterns.html) by [Agentic AI @ KFUPM](https://agenticaikfupm.github.io/).

A task is split into fixed steps. Each step's output becomes the next step's
input, and a "gate" between steps can stop the chain early if a check fails.
Because the path is fixed in code, the behavior is predictable and easy to debug.

Illustrative: each step would normally be its own LLM call with its own prompt.

Run each cell with **Shift+Enter**. Edit anything and run it again to experiment.

In [ ]:
def llm(task: str, text: str) -> str:
    """Stand-in for an LLM; each 'task' would be a different prompt."""
    if task == "extract":
        return "smart parking; real-time occupancy sensors; mobile app; reduce search time"
    if task == "outline":
        points = [p.strip() for p in text.split(";")]
        return "\n".join(f"{i}. {p.capitalize()}" for i, p in enumerate(points, 1))
    if task == "write":
        return ("Our campus smart-parking pilot uses real-time occupancy sensors and a "
                "mobile app so drivers spend less time searching for a space.")
    raise ValueError(task)


def gate_has_enough_points(outline: str, minimum: int = 3) -> bool:
    return len(outline.splitlines()) >= minimum


def sequential_pipeline(notes: str) -> str:
    print("Step 1: extract key ideas")
    ideas = llm("extract", notes)
    print("   ->", ideas)

    print("Step 2: turn ideas into an outline")
    outline = llm("outline", ideas)
    print("   ->", outline.replace("\n", "\n      "))

    print("Gate: does the outline have at least 3 points?")
    if not gate_has_enough_points(outline):
        return "Stopped: not enough material. Ask the user for more notes."
    print("   -> yes, continue")

    print("Step 3: write the final paragraph")
    return llm("write", outline)


notes = "Meeting notes: parking is hard near Building 22; sensors could help; app idea..."
print("\nFinal output:", sequential_pipeline(notes))

## Optional: try a real open model

The example above uses a small stand-in function so it runs anywhere. The cell
below loads a small open instruction-tuned model from Hugging Face (no API key).
Use `real_llm(prompt)` in place of the stand-in to see how a real model behaves.
The first run downloads about 1 GB.

In [ ]:
!pip -q install transformers accelerate
from transformers import pipeline

chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

def real_llm(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    out = chat(messages, max_new_tokens=200)
    return out[0]["generated_text"][-1]["content"]

print(real_llm("In two sentences, what is an AI agent?"))